# 🐖 17 — Auditoría de adquisición, sesiones y tamaños (CPU)

**Propósito:** corregir la descripción de la partición temporal y cuantificar las dos familias nominales de imágenes. Documenta dependencia potencial entre fotogramas con el mismo prefijo temporal y calcula terciles de tamaños normalizados a partir de GT de validación. No confundir la familia en el nombre con un dominio externo confirmado; no generar un falso reclamo de generalización entre granjas.

## 1. Dependencias CPU

In [ ]:
%pip -q install "pandas>=2,<3" numpy "openpyxl>=3.1" matplotlib

## 2. Confirmación de datos ya disponibles en Drive

In [ ]:
from google.colab import drive,files
from pathlib import Path
from collections import Counter,defaultdict
from datetime import datetime,timezone
import json,zipfile,hashlib,math
import numpy as np,pandas as pd,matplotlib.pyplot as plt
drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/Proyecto de investigación: Visión Artificial y Deep Learning en Cerdos 🐖')
assert ROOT.is_dir(), f'No encuentro {ROOT}; revisa la cuenta de Google'
REPORT=ROOT/'03_Resultados'/'05_Reportes'; METRIC=ROOT/'03_Resultados'/'03_Metricas'
FIG=ROOT/'03_Resultados'/'04_Figuras'; EXPORT=ROOT/'05_Exportaciones_ZIP'
for d in (REPORT,METRIC,FIG,EXPORT):d.mkdir(parents=True,exist_ok=True)
D09=REPORT/'CERDOS_09_COCO_Armonizado';D13=REPORT/'CERDOS_13_Calibracion_Validacion'
Y='YOLO11n-seg';M='Mask R-CNN'
def readj(p):
    assert p.is_file(),f'No encuentro: {p}'
    return json.loads(p.read_text(encoding='utf8'))
GT={'val':readj(D13/'CERDOS_13_GT_val_coco.json'),'test':readj(D09/'CERDOS_09_GT_test_coco.json')}
PREDS={
 ('val',Y):readj(D13/'CERDOS_13_Pred_YOLO_val_coco.json'),
 ('val',M):readj(D13/'CERDOS_13_Pred_MaskRCNN_val_coco.json'),
 ('test',Y):readj(D09/'CERDOS_09_Pred_YOLO_coco.json'),
 ('test',M):readj(D09/'CERDOS_09_Pred_MaskRCNN_coco.json')}
assert len(GT['val']['images'])==196 and len(GT['test']['images'])==373
assert len(GT['val']['annotations'])==1995 and len(GT['test']['annotations'])==4648
P09=readj(D09/'CERDOS_09_Protocolo.json');P13=readj(D13/'CERDOS_13_Protocolo.json')
for key in ('yaml_sha256','yolo_sha256','mask_sha256'):
    assert P09[key]==P13[key],f'Protocolos distintos: {key}'
print('✅ Predicciones previas y pesos verificados. No se necesita GPU ni reentrenar.')


## 3. Auditoría de etiquetas de adquisición y prefijos temporales

In [ ]:
# No presuponer que all_in_one y left_cropped son granjas ni cámaras.
manifest=readj(ROOT/'01_Datasets'/'FaroPigSeg'/'split_fecha_v1'/'particion_fecha_manifest.json')
assign=manifest['asignaciones_fecha']
L=[]
for split in ('train','val','test'):
 path=ROOT/'01_Datasets'/'FaroPigSeg'/'split_fecha_v1'/f'{split}.txt'
 assert path.is_file(),f'Falta lista particion: {path}'
 for raw in path.read_text(encoding='utf8').splitlines():
    if not raw.strip():continue
    name=Path(raw.strip()).name
    date=name[:10]
    assert assign[date]==split,f'Fecha mal asignada: {name}'
    source='all_in_one' if 'all_in_one' in name else 'left_cropped' if 'left_cropped' in name else 'otra'
    # La hora del nombre es un indicio de agrupación, no una identidad de video confirmada.
    group=name[:19] if len(name)>=19 else name[:10]
    L.append({'split':split,'archivo':name,'fecha':date,'timestamp_prefijo':group,
              'familia_nombre':source})
F=pd.DataFrame(L)
assert len(F)==1518 and F.archivo.nunique()==1518,'Duplicados o imágenes faltantes'
assert len(F[F.split=='train'])==949 and len(F[F.split=='val'])==196 and len(F[F.split=='test'])==373
assert F.groupby('fecha')['split'].nunique().max()==1,'¡Fechas compartidas!'
S=F.groupby(['split','familia_nombre'],as_index=False).agg(
   imagenes=('archivo','size'),fechas=('fecha','nunique'),prefijos_hora=('timestamp_prefijo','nunique'))
DATE=F.groupby(['split','fecha','familia_nombre'],as_index=False).agg(
    imagenes=('archivo','size'),prefijos_hora=('timestamp_prefijo','nunique'))
SESS=F.groupby(['split','familia_nombre','timestamp_prefijo'],as_index=False).agg(
    imagenes=('archivo','size'),fechas=('fecha','nunique'))
print('✅ Familias por nombres de archivo (NO dominios externos verificados):')
display(S)
print('✅ Prefijos timestamp con múltiples imágenes:')
display(SESS.sort_values('imagenes',ascending=False).head(12))


## 4. Terciles GT, Excel y ZIP

In [ ]:
# Los terciles describen tamaño de máscara GT normalizado por área de imagen.
obj=[]
for split in ('val','test'):
    gt=GT[split];images={int(i['id']):i for i in gt['images']}
    for a in gt['annotations']:
        im=images[int(a['image_id'])]
        frac=float(a['area'])/(int(im['height'])*int(im['width']))
        obj.append({'split':split,'fecha':im['file_name'][:10],
                    'familia':'all_in_one' if 'all_in_one' in im['file_name'] else 'left_cropped',
                    'imagen':im['file_name'],'GT_area_px2':float(a['area']),
                    'fraccion_area_imagen':frac})
OBJ=pd.DataFrame(obj)
qs=OBJ.loc[OBJ.split=='val','fraccion_area_imagen'].quantile([1/3,2/3]).to_list()
assert 0<qs[0]<qs[1]<1,'Terciles de validación inválidos'
OBJ['tercil_ref_val']=pd.cut(OBJ.fraccion_area_imagen,bins=[-np.inf,qs[0],qs[1],np.inf],labels=['small','medium','large'])
OBJ_STATS=OBJ.groupby(['split','tercil_ref_val'],observed=True,as_index=False).agg(n_instancias=('imagen','size'),area_mediana_rel=('fraccion_area_imagen','median'))
# Evitar interpretar diferencias descriptivas como significación estadística o traslado de dominio.
OUT=METRIC/'CERDOS_17_Auditoria_Fuentes_Sesiones.xlsx'
with pd.ExcelWriter(OUT,engine='openpyxl') as w:
    S.to_excel(w,sheet_name='Familias',index=False)
    DATE.to_excel(w,sheet_name='Fechas',index=False)
    SESS.to_excel(w,sheet_name='Prefijos_timestamp',index=False)
    F.to_excel(w,sheet_name='Manifest_1518',index=False)
    OBJ_STATS.to_excel(w,sheet_name='Terciles_resumen',index=False)
    OBJ.to_excel(w,sheet_name='Objetos_val_test',index=False)
    pd.DataFrame([{'nota':s} for s in ['Las familias son cadenas de nombres, no metadata verificada de cámara/granja.',
               'Las fechas provienen de nombres; prefijos timestamp no prueban un único video.',
               'Terciles de área fueron fijados SOLO usando GT de validación; descriptivos, no AP por tercil.',
               'Sin validación externa; los mismos dominios nominales aparecen en train/val/test.',
               'Para experimento fuente A a B se requiere un split completamente nuevo antes del entrenamiento.']]).to_excel(w,sheet_name='Advertencias',index=False)
fig,ax=plt.subplots(figsize=(9,5))
X=S.pivot(index='split',columns='familia_nombre',values='imagenes').reindex(['train','val','test'])
X.plot.bar(ax=ax,color=['#648BAB','#D28A73'],rot=0)
ax.set(xlabel='Partition',ylabel='Images',title='Image naming families across temporal partitions');ax.legend(title='Filename family')
fig.tight_layout();FIGURE=FIG/'CERDOS_17_Familias_Adquisicion.png';fig.savefig(FIGURE,dpi=400);plt.close(fig)
J=REPORT/'CERDOS_17_Auditoria_Fuentes_Sesiones.json'
J.write_text(json.dumps({'version':'17_name_family_audit_v1','timestamp_utc':datetime.now(timezone.utc).isoformat(),
                         'familias':S.to_dict('records'),'terciles_GT_val':qs,
                         'notas':['Fuentes nominales no prueban cámaras, granjas ni dominios independientes.',
                         'Sin nueva validación externa; sin nuevo entrenamiento ni model selection sobre test.']},ensure_ascii=False,indent=2),encoding='utf8')
ZIP=EXPORT/'CERDOS_17_Auditoria_Fuentes_Sesiones.zip'
with zipfile.ZipFile(ZIP,'w',compression=zipfile.ZIP_DEFLATED) as z:
 for p in (OUT,FIGURE,J):z.write(p,arcname=p.name)
assert zipfile.ZipFile(ZIP).testzip() is None
print('✅ 17 guardado:',ZIP)
files.download(str(ZIP))
